# ⚡ PHANTOM Cloud Hardware Testbed & Zero-Disk Benchmark

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FreakyAdy/phantom/blob/main/notebooks/phantom_cloud_tester.ipynb)

This notebook enables **100% free, zero-local-disk evaluation and reproduction** of the entire PHANTOM 14-model benchmark suite (spanning 135M to 72.7B parameters):
- **Free 15 GB Nvidia Cloud GPU** (T4 / L4)
- **100 GB Fast Ephemeral Cloud SSD** (0 bytes consumed on your laptop)
- **12.7 GB Host RAM**
- **1-Click Verification Across All 14 Evaluated Scale Models** (Qwen, DeepSeek, Llama, Mixtral, QwQ, Command-R, Yi)
- **Automated Publication-Ready Markdown & JSON Report Generation** (matching `docs/testing/TEMPLATE_TEST_REPORT.md`)


### Step 1: Environment Setup & Cloud Hardware Inspection
Clones the official PHANTOM repository, installs dependencies, and inspects live cloud GPU telemetry.

In [ ]:
# Clone repository if running inside Google Colab
import os, sys
if not os.path.exists('/content/phantom'):
    !git clone https://github.com/FreakyAdy/phantom.git /content/phantom
    %cd /content/phantom
else:
    %cd /content/phantom
    !git pull

# Install dependencies
!pip install -q -e python --no-deps
!pip install -q rich structlog huggingface_hub

# Hardware inspection
import torch, shutil
print('=' * 68)
print('  ⚡ PHANTOM CLOUD TESTBED — HARDWARE STATUS')
print('=' * 68)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'  ✓ Cloud GPU:       {gpu_name} ({vram_gb:.1f} GB VRAM)')
    print(f'  ✓ CUDA Version:    {torch.version.cuda}')
else:
    print('  ⚠ No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU.')

total_b, used_b, free_b = shutil.disk_usage('/content')
print(f'  ✓ Cloud Disk:      {free_b / (1024**3):.1f} GB free on ephemeral SSD')
print('=' * 68)


### Step 2: Interactive Model Selector & Hardware Profiler (1-Click Form)
Select any of the 14 verified models and target hardware configuration.
You can choose **Instant Virtual Profile** (simulates layer tiering and tok/s in milliseconds without downloading weights) or **Live Cloud Inference** (downloads model to ephemeral cloud scratch).

In [ ]:
#@title 🛠️ Select Model & Hardware Preset { run: "auto" }
MODEL_NAME = "qwen3-30b-a3b" #@param ["smollm-135m", "qwen2.5-coder-32b", "qwen3-30b-a3b", "llama-3-70b", "deepseek-r1-distill-qwen-32b", "deepseek-r1-distill-llama-70b", "mixtral-8x7b-instruct", "qwq-32b-preview", "qwen2.5-72b-instruct", "qwen2.5-32b-instruct", "deepseek-coder-33b-instruct", "codellama-70b-instruct", "command-r-35b", "yi-1.5-34b-chat"]
HARDWARE_PRESET = "colab-t4" #@param ["colab-t4", "rtx4050-laptop", "rtx4070-desktop", "rtx4090-desktop", "apple-m3-pro"]
MODE = "Virtual Simulation (Instant Zero-Disk Profile)" #@param ["Virtual Simulation (Instant Zero-Disk Profile)", "Live Cloud Inference (Download to Ephemeral Scratch)"]

print(f"Selected Model:    {MODEL_NAME}")
print(f"Hardware Preset:   {HARDWARE_PRESET}")
print(f"Execution Mode:    {MODE}")

# Run zero-disk profiler
!phantom profile {MODEL_NAME} --preset {HARDWARE_PRESET}


### Step 3: Run Automated Verification Battery & Generate Formal Report
Executes the automated verification battery:
1. **Algorithmic Dynamic Programming** (0/1 Knapsack, target 220)
2. **Mathematical Deduction** (Harmonic Mean Velocity, target 48 mph)
3. **Code Synthesis & Whitespace Normalization** (Word reversal)

Generates both a raw JSON telemetry record and a publication-ready Markdown report matching `docs/testing/TEMPLATE_TEST_REPORT.md`.

In [ ]:
import os
from pathlib import Path
from IPython.display import Markdown, display

is_dry = "Instant" in MODE
dry_flag = "--dry-run" if is_dry else ""

# Execute cloud runner harness
!python scripts/colab_runner.py --model {MODEL_NAME} --preset {HARDWARE_PRESET} {dry_flag} --output-dir /content/reports

# Display generated markdown report inline in notebook
reports_dir = Path("/content/reports")
md_reports = list(reports_dir.glob("*.md"))
if md_reports:
    latest_md = sorted(md_reports, key=lambda p: p.stat().st_mtime)[-1]
    print("=" * 68)
    print(f"  DISPLAYING GENERATED REPORT: {latest_md.name}")
    print("=" * 68)
    display(Markdown(latest_md.read_text(encoding="utf-8")))


### Step 4: Export Reports & Purge Cloud Scratch
Downloads the Markdown report and raw JSON results directly to your local computer,
then deletes any temporary model weights from Colab scratch drive (0 bytes remaining).

### Step 5: PHANTOM v2 MD Blueprint — Live E2E + MoE Prefetch Correlation
Runs the full v2 Colab test suite:
1. **EAGLE-3 head training** on cloud GPU (ephemeral scratch)
2. **MoE expert routing / Wraith prefetch correlation** (Qwen3-30B-A3B profile)
3. **Live v2 speculative decode** with `--live` (downloads GGUF to `/content/scratch`)
4. **v2 ablation benchmark** + merge into `benchmarks/results/v2_latest.json`

Select **Live Cloud Inference** in Step 2 to download weights; use **Virtual Simulation** for instant dry-run validation.

In [ ]:
#@title ⚡ PHANTOM v2 Colab Test Suite { run: "auto" }
V2_MODEL = "qwen2.5-coder-32b"  #@param ["qwen2.5-coder-32b", "qwen3-30b-a3b"]
V2_PRESET = "colab-t4"  #@param ["colab-t4", "rtx4050-laptop"]

is_dry = "Instant" in MODE
live_flag = "" if is_dry else "--live"
dry_flag = "--dry-run" if is_dry else ""

print(f"v2 Model: {V2_MODEL} | Preset: {V2_PRESET} | Live: {not is_dry}")

# Run full v2 Colab harness
!python scripts/colab_v2_runner.py --model {V2_MODEL} --preset {V2_PRESET} {live_flag} {dry_flag} --output-dir /content/reports

# MoE prefetch correlation (Qwen3 profile)
!python scripts/colab_v2_runner.py --moe-correlation

# Regenerate RESULTS.md with v2 section
!python scripts/generate_results.py

# Display v2 report inline
from pathlib import Path
from IPython.display import Markdown, display

reports_dir = Path("/content/reports")
v2_reports = sorted(reports_dir.glob("test_16_phantom_v2_colab.md"))
if v2_reports:
    latest_v2 = v2_reports[-1]
    print("=" * 68)
    print(f"  DISPLAYING v2 REPORT: {latest_v2.name}")
    print("=" * 68)
    display(Markdown(latest_v2.read_text(encoding="utf-8")))
else:
    print("No v2 report found — check runner output above.")

In [ ]:
import shutil
try:
    from google.colab import files
    is_colab = True
except ImportError:
    is_colab = False

reports_dir = Path("/content/reports")
if reports_dir.exists():
    for report_file in sorted(reports_dir.glob("*")):
        print(f"Ready for download: {report_file.name} ({report_file.stat().st_size} bytes)")
        if is_colab:
            files.download(str(report_file))

# Reclaim scratch disk
if os.path.exists("/content/scratch"):
    shutil.rmtree("/content/scratch")
    print("✓ Ephemeral cloud scratch drive cleaned. 0 bytes remaining on disk.")
else:
    print("✓ Clean environment: zero model weight files on disk.")
